# 01 — Task 2.1: PII Masking & Tokenization (Silver Layer)

**Objective:** Read Bronze Delta tables, apply PII masking/tokenization, and write PII-safe Silver tables.  
**Compliance:** PCI-DSS, GDPR Article 89, RBI KYC Guidelines.

| Field | Rule | Example |
|-------|------|---------|
| `card_number` | BIN (first 6) + `XXXXXX` + last 4 | `411111XXXXXX1234` |
| `pan_number` | HMAC-SHA-256 with secret from vault | `a3f8c2...` (hex) |
| `mobile_number` | Keep first 2 + `XXXXXX` + last 2 | `98XXXXXX12` |
| `email` | Keep first 2 of local + `****@domain` | `jo****@gmail.com` |
| `full_name` | Pseudonymize (deterministic hash-based) | Consistent alias |
| `ip_address` | Truncate last octet → `.0` | `192.168.1.0` |

> **Bronze layer remains append-only and untouched** (7-year regulatory retention).

---
## Cell 1 — Configuration & Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import hashlib
import hmac
from datetime import datetime

# ── Catalog / Schema Configuration ──────────────────────────────────────
CATALOG       = "bfsi"
BRONZE_SCHEMA = "bronze_layer"
SILVER_SCHEMA = "silver_layer"

# ── Bronze source tables ───────────────────────────────────────────────
CARD_TXN_TABLE   = f"{CATALOG}.{BRONZE_SCHEMA}.card_transactions"
CUSTOMERS_TABLE  = f"{CATALOG}.{BRONZE_SCHEMA}.customers"
UPI_TXN_TABLE    = f"{CATALOG}.{BRONZE_SCHEMA}.upi_transactions"
NEFT_RTGS_TABLE  = f"{CATALOG}.{BRONZE_SCHEMA}.neft_rtgs_transactions"

# ── Silver target tables ────────────────────────────────────────────────
SILVER_CARD_TXN   = f"{CATALOG}.{SILVER_SCHEMA}.card_transactions_masked"
SILVER_CUSTOMERS  = f"{CATALOG}.{SILVER_SCHEMA}.customers_masked"
SILVER_UPI_TXN    = f"{CATALOG}.{SILVER_SCHEMA}.upi_transactions_masked"
SILVER_NEFT_RTGS  = f"{CATALOG}.{SILVER_SCHEMA}.neft_rtgs_transactions_clean"

print(f"Bronze → Silver pipeline initialized")
print(f"Timestamp: {datetime.now().isoformat()}")

---
## Cell 2 — Secret Keys (Hardcoded)

In [0]:
# ── Secret Keys (Hardcoded) ───────────────────────────────────────────
# TODO: Replace these hardcoded keys with Databricks Secret Vault:
#       HMAC_SECRET_KEY = dbutils.secrets.get(scope="bfsi_secrets", key="hmac_secret_key")
#       PSEUDONYM_SALT  = dbutils.secrets.get(scope="bfsi_secrets", key="pseudonym_salt")

HMAC_SECRET_KEY = "bfsi-fraud-detection-hmac-key-2025-SECURE"
PSEUDONYM_SALT  = "bfsi-pseudonym-salt-2025-CHANGE-IN-PROD"

print(f"HMAC key loaded       : {'*' * 20} (length={len(HMAC_SECRET_KEY)})")
print(f"Pseudonym salt loaded : {'*' * 20} (length={len(PSEUDONYM_SALT)})")
print("Secrets ready for PII masking.")

---
## Cell 3 — Create Silver Schema

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
    COMMENT 'Silver layer — PII-masked, cleansed, and conformed data'
""")
print(f"Schema {CATALOG}.{SILVER_SCHEMA} ready.")

---
## Cell 3a — Incremental Load Helper (Watermark Lookup)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  INCREMENTAL PROCESSING HELPER
#  Reads the max(_silver_load_ts) from the silver table to determine
#  which bronze rows are new and need processing.
#  On first run (table doesn't exist), processes all data.
# ══════════════════════════════════════════════════════════════════════════

from datetime import datetime as _dt

def get_last_load_ts(silver_table_name):
    """Get the max _silver_load_ts from a silver table.
    Returns a timestamp string. If the table doesn't exist or is empty,
    returns '1900-01-01 00:00:00' to process all data.
    """
    try:
        max_ts = (
            spark.read.table(silver_table_name)
            .agg(F.max("_silver_load_ts").alias("max_ts"))
            .collect()[0]["max_ts"]
        )
        if max_ts is not None:
            print(f"  Last load timestamp for {silver_table_name}: {max_ts}")
            return max_ts
    except Exception:
        print(f"  Table {silver_table_name} does not exist yet — will process all data.")
    return _dt(1900, 1, 1)

print("Incremental load helper ready.")

---
## Cell 4 — Define PII Masking UDFs

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  PII MASKING FUNCTIONS — registered as Spark UDFs
# ══════════════════════════════════════════════════════════════════════════

# ── 1. Card Number Masking (PCI-DSS compliant) ─────────────────────────
#    Rule: Keep first 6 (BIN) + XXXXXX + last 4
#    Example: 4111111111111234 → 411111XXXXXX1234

def mask_card_number(card_number):
    """Mask card number: preserve BIN (first 6) and last 4 digits."""
    if card_number is None or len(card_number) < 10:
        return card_number
    # Remove any spaces/dashes for uniform processing
    clean = card_number.replace(" ", "").replace("-", "")
    if len(clean) < 10:
        return card_number
    bin_part  = clean[:6]           # First 6 = BIN (Bank Identification Number)
    last_four = clean[-4:]          # Last 4 = visible to cardholder
    masked    = "X" * (len(clean) - 10)  # Middle digits masked
    return f"{bin_part}{masked}{last_four}"

mask_card_number_udf = F.udf(mask_card_number, StringType())


# ── 2. PAN Number Tokenization (HMAC-SHA-256) ──────────────────────────
#    Rule: HMAC-SHA-256 hash using secret key from vault
#    Result: deterministic 64-char hex digest (irreversible)

def hash_pan_number(pan_number):
    """Tokenize PAN via HMAC-SHA-256 with secret key."""
    if pan_number is None:
        return None
    # Normalize: uppercase, strip whitespace
    pan_clean = pan_number.strip().upper()
    # HMAC-SHA-256 with the secret key
    token = hmac.new(
        key=HMAC_SECRET_KEY.encode("utf-8"),
        msg=pan_clean.encode("utf-8"),
        digestmod=hashlib.sha256
    ).hexdigest()
    return token

hash_pan_number_udf = F.udf(hash_pan_number, StringType())


# ── 3. Mobile Number Masking ───────────────────────────────────────────
#    Rule: Keep first 2 digits + XXXXXX + last 2 digits
#    Example: 9876543210 → 98XXXXXX10

def mask_mobile_number(mobile):
    """Mask mobile number: keep first 2 + XXXXXX + last 2."""
    if mobile is None:
        return None
    # Strip country code prefix if present (e.g., +91)
    clean = mobile.replace(" ", "").replace("-", "")
    if clean.startswith("+"):
        clean = clean[3:]  # Remove +91 or similar
    if len(clean) < 4:
        return "XXXX"
    first_two = clean[:2]
    last_two  = clean[-2:]
    middle    = "X" * (len(clean) - 4)
    return f"{first_two}{middle}{last_two}"

mask_mobile_udf = F.udf(mask_mobile_number, StringType())


# ── 4. Email Masking ───────────────────────────────────────────────────
#    Rule: Keep first 2 chars of local part + **** + @domain
#    Example: john.doe@gmail.com → jo****@gmail.com

def mask_email(email):
    """Mask email: keep first 2 chars of local part + mask + domain."""
    if email is None:
        return None
    if "@" not in email:
        return "****"
    local, domain = email.rsplit("@", 1)
    if len(local) <= 2:
        masked_local = local + "****"
    else:
        masked_local = local[:2] + "****"
    return f"{masked_local}@{domain}"

mask_email_udf = F.udf(mask_email, StringType())


# ── 5. Full Name Pseudonymization ──────────────────────────────────────
#    Rule: Deterministic hash-based pseudonym using salted SHA-256
#    Same name → always same pseudonym (for join consistency)

# Pseudonym lookup pool — Indian names for BFSI context
_PSEUDONYM_FIRST_NAMES = [
    "Aarav", "Vivaan", "Aditya", "Vihaan", "Arjun",
    "Sai", "Reyansh", "Ayaan", "Krishna", "Ishaan",
    "Ananya", "Diya", "Myra", "Sara", "Aadhya",
    "Isha", "Kavya", "Riya", "Priya", "Neha",
    "Rohan", "Karan", "Amit", "Suresh", "Rahul",
    "Pooja", "Sneha", "Meera", "Lakshmi", "Deepa",
    "Vikram", "Raj", "Dev", "Nikhil", "Sanjay",
    "Geeta", "Sunita", "Rekha", "Nandini", "Swati"
]

_PSEUDONYM_LAST_NAMES = [
    "Sharma", "Verma", "Patel", "Gupta", "Singh",
    "Kumar", "Reddy", "Nair", "Joshi", "Mehta",
    "Iyer", "Rao", "Das", "Mishra", "Chauhan",
    "Pillai", "Menon", "Bhat", "Desai", "Shah",
    "Tiwari", "Pandey", "Saxena", "Agarwal", "Malhotra",
    "Kulkarni", "Sinha", "Thakur", "Banerjee", "Mukherjee"
]


def pseudonymize_name(full_name):
    """Generate a deterministic pseudonym from the original name."""
    if full_name is None:
        return None
    # Salted SHA-256 hash of the name
    salted = f"{PSEUDONYM_SALT}:{full_name.strip().lower()}"
    hash_val = hashlib.sha256(salted.encode("utf-8")).hexdigest()
    # Use hash to deterministically pick first + last name
    first_idx = int(hash_val[:8], 16) % len(_PSEUDONYM_FIRST_NAMES)
    last_idx  = int(hash_val[8:16], 16) % len(_PSEUDONYM_LAST_NAMES)
    return f"{_PSEUDONYM_FIRST_NAMES[first_idx]} {_PSEUDONYM_LAST_NAMES[last_idx]}"

pseudonymize_name_udf = F.udf(pseudonymize_name, StringType())


# ── 6. IP Address Truncation (GDPR Article 89) ─────────────────────────
#    Rule: Replace last octet with 0
#    Example: 192.168.1.45 → 192.168.1.0

def truncate_ip_address(ip):
    """Truncate IP last octet for GDPR compliance."""
    if ip is None:
        return None
    parts = ip.split(".")
    if len(parts) != 4:
        return ip  # Not a valid IPv4, return as-is
    parts[-1] = "0"
    return ".".join(parts)

truncate_ip_udf = F.udf(truncate_ip_address, StringType())


print("All 6 PII masking UDFs registered successfully.")
print("  1. mask_card_number_udf   — Card PAN masking (BIN + last 4)")
print("  2. hash_pan_number_udf    — PAN HMAC-SHA-256 tokenization")
print("  3. mask_mobile_udf        — Mobile number masking")
print("  4. mask_email_udf         — Email address masking")
print("  5. pseudonymize_name_udf  — Full name pseudonymization")
print("  6. truncate_ip_udf        — IP address truncation")

---
## Cell 5 — Silver Table 1: Card Transactions (Masked)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  SILVER TABLE 1: card_transactions_masked
#  Source: bfsi.bronze_layer.card_transactions
#  PII field masked: card_number
#  Mode: INCREMENTAL — only processes new Bronze rows
# ══════════════════════════════════════════════════════════════════════════

last_load_ts = get_last_load_ts(SILVER_CARD_TXN)

print(f"Reading Bronze table: {CARD_TXN_TABLE}")
df_card_bronze = (
    spark.read.table(CARD_TXN_TABLE)
    .filter(F.col("_load_ts") > F.lit(last_load_ts))
)

new_count = df_card_bronze.count()
print(f"New Bronze records to process: {new_count:,}")

if new_count > 0:
    # Apply card_number masking
    df_card_silver = (
        df_card_bronze
        # ── PII masking ──
        .withColumn("card_number_masked", mask_card_number_udf(F.col("card_number")))
        .drop("card_number")
        .withColumnRenamed("card_number_masked", "card_number")
        # ── Silver metadata ──
        .withColumn("_silver_load_ts", F.current_timestamp())
        .withColumn("_pii_masking_applied", F.lit(True))
        .withColumn("_masking_version", F.lit("v1.0-task2.1"))
    )

    # Append new records to Silver Delta table
    df_card_silver.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(SILVER_CARD_TXN)

    print(f"\n Silver table updated: {SILVER_CARD_TXN}")
    print(f"New records appended: {new_count:,}")
    print(f"Total Silver records: {spark.read.table(SILVER_CARD_TXN).count():,}")
else:
    print(f"No new data found. Skipping {SILVER_CARD_TXN}.")

---
## Cell 6 — Silver Table 2: Customers (Masked)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  SILVER TABLE 2: customers_masked
#  Source: bfsi.bronze_layer.customers
#  PII fields masked: pan_number, mobile_number, email, full_name
#  Mode: INCREMENTAL — only processes new Bronze rows
# ══════════════════════════════════════════════════════════════════════════

last_load_ts = get_last_load_ts(SILVER_CUSTOMERS)

print(f"Reading Bronze table: {CUSTOMERS_TABLE}")
df_customers_bronze = (
    spark.read.table(CUSTOMERS_TABLE)
    .filter(F.col("_load_ts") > F.lit(last_load_ts))
)

new_count = df_customers_bronze.count()
print(f"New Bronze records to process: {new_count:,}")

if new_count > 0:
    # Apply all customer PII masking
    df_customers_silver = (
        df_customers_bronze
        # ── PAN number: HMAC-SHA-256 tokenization ──
        .withColumn("pan_number_tokenized", hash_pan_number_udf(F.col("pan_number")))
        .drop("pan_number")
        .withColumnRenamed("pan_number_tokenized", "pan_number")
        # ── Mobile number: mask middle digits ──
        .withColumn("mobile_masked", mask_mobile_udf(F.col("mobile_number")))
        .drop("mobile_number")
        .withColumnRenamed("mobile_masked", "mobile_number")
        # ── Email: mask local part ──
        .withColumn("email_masked", mask_email_udf(F.col("email")))
        .drop("email")
        .withColumnRenamed("email_masked", "email")
        # ── Full name: pseudonymize ──
        .withColumn("name_pseudo", pseudonymize_name_udf(F.col("full_name")))
        .drop("full_name")
        .withColumnRenamed("name_pseudo", "full_name")
        # ── Silver metadata ──
        .withColumn("_silver_load_ts", F.current_timestamp())
        .withColumn("_pii_masking_applied", F.lit(True))
        .withColumn("_masking_version", F.lit("v1.0-task2.1"))
    )

    # Append new records to Silver Delta table
    df_customers_silver.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(SILVER_CUSTOMERS)

    print(f"\n Silver table updated: {SILVER_CUSTOMERS}")
    print(f"New records appended: {new_count:,}")
    print(f"Total Silver records: {spark.read.table(SILVER_CUSTOMERS).count():,}")
else:
    print(f"No new data found. Skipping {SILVER_CUSTOMERS}.")

---
## Cell 7 — Silver Table 3: UPI Transactions (Masked)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  SILVER TABLE 3: upi_transactions_masked
#  Source: bfsi.bronze_layer.upi_transactions
#  PII field masked: ip_address
#  Mode: INCREMENTAL — only processes new Bronze rows
# ══════════════════════════════════════════════════════════════════════════

last_load_ts = get_last_load_ts(SILVER_UPI_TXN)

print(f"Reading Bronze table: {UPI_TXN_TABLE}")
df_upi_bronze = (
    spark.read.table(UPI_TXN_TABLE)
    .filter(F.col("_load_ts") > F.lit(last_load_ts))
)

new_count = df_upi_bronze.count()
print(f"New Bronze records to process: {new_count:,}")

if new_count > 0:
    # Apply IP address truncation
    df_upi_silver = (
        df_upi_bronze
        # ── IP address: truncate last octet (GDPR Article 89) ──
        .withColumn("ip_truncated", truncate_ip_udf(F.col("ip_address")))
        .drop("ip_address")
        .withColumnRenamed("ip_truncated", "ip_address")
        # ── Silver metadata ──
        .withColumn("_silver_load_ts", F.current_timestamp())
        .withColumn("_pii_masking_applied", F.lit(True))
        .withColumn("_masking_version", F.lit("v1.0-task2.1"))
    )

    # Append new records to Silver Delta table
    df_upi_silver.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(SILVER_UPI_TXN)

    print(f"\n Silver table updated: {SILVER_UPI_TXN}")
    print(f"New records appended: {new_count:,}")
    print(f"Total Silver records: {spark.read.table(SILVER_UPI_TXN).count():,}")
else:
    print(f"No new data found. Skipping {SILVER_UPI_TXN}.")

---
## Cell 8 — Silver Table 4: NEFT/RTGS Transactions (Clean)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  SILVER TABLE 4: neft_rtgs_transactions_clean
#  Source: bfsi.bronze_layer.neft_rtgs_transactions
#  PII fields masked: debtor_name, creditor_name (pseudonymized)
#  Mode: INCREMENTAL — only processes new Bronze rows
# ══════════════════════════════════════════════════════════════════════════

last_load_ts = get_last_load_ts(SILVER_NEFT_RTGS)

print(f"Reading Bronze table: {NEFT_RTGS_TABLE}")
df_neft_bronze = (
    spark.read.table(NEFT_RTGS_TABLE)
    .filter(F.col("_load_ts") > F.lit(last_load_ts))
)

new_count = df_neft_bronze.count()
print(f"New Bronze records to process: {new_count:,}")

if new_count > 0:
    # Apply name pseudonymization
    df_neft_silver = (
        df_neft_bronze
        # ── Debtor name: pseudonymize ──
        .withColumn("debtor_pseudo", pseudonymize_name_udf(F.col("debtor_name")))
        .drop("debtor_name")
        .withColumnRenamed("debtor_pseudo", "debtor_name")
        # ── Creditor name: pseudonymize ──
        .withColumn("creditor_pseudo", pseudonymize_name_udf(F.col("creditor_name")))
        .drop("creditor_name")
        .withColumnRenamed("creditor_pseudo", "creditor_name")
        # ── Silver metadata ──
        .withColumn("_silver_load_ts", F.current_timestamp())
        .withColumn("_pii_masking_applied", F.lit(True))
        .withColumn("_masking_version", F.lit("v1.0-task2.1"))
    )

    # Append new records to Silver Delta table
    df_neft_silver.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(SILVER_NEFT_RTGS)

    print(f"\n Silver table updated: {SILVER_NEFT_RTGS}")
    print(f"New records appended: {new_count:,}")
    print(f"Total Silver records: {spark.read.table(SILVER_NEFT_RTGS).count():,}")
else:
    print(f"No new data found. Skipping {SILVER_NEFT_RTGS}.")

---
## Cell 9 — Summary & Data Quality Report

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  TASK 2.1 COMPLETION SUMMARY
# ══════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  TASK 2.1: PII MASKING & TOKENIZATION — COMPLETE")
print("=" * 70)

# Record counts for each Silver table
silver_tables = [
    (SILVER_CARD_TXN,  "Card Transactions"),
    (SILVER_CUSTOMERS, "Customers"),
    (SILVER_UPI_TXN,   "UPI Transactions"),
    (SILVER_NEFT_RTGS, "NEFT/RTGS Transactions"),
]

print(f"\n{'Table':<50} {'Records':>12}")
print("-" * 64)
total = 0
for table_name, label in silver_tables:
    count = spark.read.table(table_name).count()
    total += count
    print(f"{label:<50} {count:>12,}")
print("-" * 64)
print(f"{'TOTAL':<50} {total:>12,}")

print("\n PII Masking Applied:")
print("  ✓ card_number  → BIN + XXXXXX + last 4  (PCI-DSS)")
print("  ✓ pan_number   → HMAC-SHA-256 tokenized (Vault-secured key)")
print("  ✓ mobile_number→ First 2 + XXXXXX + last 2")
print("  ✓ email        → First 2 + **** + @domain")
print("  ✓ full_name    → Pseudonymized (deterministic)")
print("  ✓ ip_address   → Last octet truncated (GDPR Art. 89)")
print("  ✓ debtor/creditor names → Pseudonymized (NEFT/RTGS)")

print("\n Bronze layer: UNTOUCHED (append-only, 7-year retention)")
print(f" Masking version: v1.0-task2.1")
print(f" Timestamp: {datetime.now().isoformat()}")
print("=" * 70)

---

### Data Lineage

```
Bronze (raw, PII present)          Silver (PII-safe)
─────────────────────────          ──────────────────────────────
card_transactions          ──→    card_transactions_masked
  └─ card_number                    └─ card_number (BIN+XXXX+last4)

customers                  ──→    customers_masked
  ├─ pan_number                     ├─ pan_number (HMAC-SHA-256)
  ├─ mobile_number                  ├─ mobile_number (masked)
  ├─ email                          ├─ email (masked)
  └─ full_name                      └─ full_name (pseudonymized)

upi_transactions           ──→    upi_transactions_masked
  └─ ip_address                     └─ ip_address (last octet → .0)

neft_rtgs_transactions     ──→    neft_rtgs_transactions_clean
  ├─ debtor_name                    ├─ debtor_name (pseudonymized)
  └─ creditor_name                  └─ creditor_name (pseudonymized)
```

> **Next steps:** Task 2.2 (Unify Transaction Sources) → Task 2.3 (Fraud Feature Engineering)